# trend_participation_guard_screen_v1

Purpose: follow `wf2024_state_coverage_diagnostic_v1` with one narrow, notebook-first screen. The prior card showed the 2024 failure is a high-trend / low-volatility underparticipation state with no sequence-risk coverage and a later 20% drawdown interval. This notebook tests a default-off trend-participation floor plus volatility/equity-drawdown guard before opening any new PPO branch.

Governance:

- **Baselines**: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `warm_start_action_cap_validation_v1` selected `seq_cap10_two_of_three`.
- **Role**: `diagnose`.
- **Target window**: warm-start `wf_2022` and `wf_2023_2024`, plus the top 2024 poor-window clusters from `wf2024_state_coverage_diagnostic_v1`.
- **Changed layer**: `model_selection` / default-off reusable counterfactual overlay. No runtime profile promotion and no actor retention from this card.
- **External reference**: Qlib workflow and Qlib paper for repeatable evaluation; Probability of Backtest Overfitting for promotion discipline; time-series momentum and momentum-crash literature for trend state risk; volatility-managed portfolios for volatility guard design; FinRL for separating environment/reward/action design.
- **Hypothesis**: a train-quantile high-trend / low-VIX state can lift low positions during 2024 rally windows, but only candidates that also keep MaxDD <= 15% are useful as training/checkpoint objectives.
- **Acceptance criteria**: every scored fold must reach `relative_return >= +30pp` and `MaxDD <= 15%`; poor-window clusters must clear participation and MaxDD gates.
- **Expected trade-off**: improved 2024 participation may increase H2 drawdown; equity/volatility guards should block that risk.
- **Failure mode**: if no row clears both participation and drawdown gates, keep the best row as a training-objective hint only.
- **Comparison-table row key**: `trend_participation_guard_screen_v1`.

Historical reflection:

- Closest success: `warm_start_action_cap_validation_v1` proved fold-local sequence caps can fix `wf_2022` MaxDD when the failure state is covered.
- Closest failure: `wf2024_state_coverage_diagnostic_v1` found `trace_target_cap_changed_days=0` and sequence coverage ~0 for the 2024 bad windows, so another sequence threshold grid would be duplicate work.
- Why this is not a duplicate: this run changes the state definition to high-trend / low-volatility participation and derives thresholds from each fold's train segment, then scores both participation and drawdown gates.


In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.app.participation_counterfactual import (
    TrendParticipationGuardConfig,
    apply_trend_participation_guard_overlay,
)
from src.soxl_quant.app.state_conditioned_objective import (
    StateConditionedObjectiveConfig,
    daily_path_metrics,
    render_state_conditioned_objective_review,
    score_state_conditioned_objective,
    slice_daily_path_metrics,
    summarize_state_conditioned_objective,
)
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, init_qlib, segment_frame
from src.soxl_quant.settings import QLIB_DATA_PATH

EXPERIMENT = 'trend_participation_guard_screen_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_WARM_START = ROOT / 'runtime' / 'model_selection' / 'card' / 'warm_start_action_cap_validation_v1'
SOURCE_STATE_DIAG = ROOT / 'runtime' / 'model_selection' / 'card' / 'wf2024_state_coverage_diagnostic_v1'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0

WALK_FORWARD_WINDOWS = {
    'wf_2022': {
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
        'daily': SOURCE_WARM_START / 'seq_cap10_two_of_three_wf_2022_daily_path.csv',
    },
    'wf_2023_2024': {
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
        'daily': SOURCE_WARM_START / 'seq_cap10_two_of_three_wf_2023_2024_daily_path.csv',
    },
}

CANDIDATE_RECIPES = [
    {
        'candidate': 'base_seq_cap10_two_of_three',
        'enabled': False,
        'note': 'Unchanged selected warm-start action-cap path.',
    },
    {
        'candidate': 'trend_q60_floor45_guard15',
        'enabled': True,
        'min_target': 0.45,
        'max_target': 0.55,
        'max_base_position': 0.35,
        'anchor_q': 0.60,
        'mom_q': 0.60,
        'trend_slope_q': 0.60,
        'trend_gap_q': 0.60,
        'vix_q': 0.55,
        'realized_vol_q': 0.80,
        'drawdown_q': 0.35,
        'max_equity_drawdown': 0.15,
        'note': 'Broad train-quantile trend state with equity guard.',
    },
    {
        'candidate': 'trend_q70_floor45_guard10',
        'enabled': True,
        'min_target': 0.45,
        'max_target': 0.55,
        'max_base_position': 0.35,
        'anchor_q': 0.70,
        'mom_q': 0.70,
        'trend_slope_q': 0.65,
        'trend_gap_q': 0.65,
        'vix_q': 0.50,
        'realized_vol_q': 0.75,
        'drawdown_q': 0.40,
        'max_equity_drawdown': 0.10,
        'note': 'Stricter low-vol trend state with tighter equity guard.',
    },
    {
        'candidate': 'trend_q70_floor55_guard15',
        'enabled': True,
        'min_target': 0.55,
        'max_target': 0.65,
        'max_base_position': 0.35,
        'anchor_q': 0.70,
        'mom_q': 0.70,
        'trend_slope_q': 0.65,
        'trend_gap_q': 0.65,
        'vix_q': 0.55,
        'realized_vol_q': 0.80,
        'drawdown_q': 0.35,
        'max_equity_drawdown': 0.15,
        'note': 'Higher participation floor with moderate equity guard.',
    },
    {
        'candidate': 'trend_q80_floor55_guard10',
        'enabled': True,
        'min_target': 0.55,
        'max_target': 0.65,
        'max_base_position': 0.30,
        'anchor_q': 0.80,
        'mom_q': 0.80,
        'trend_slope_q': 0.75,
        'trend_gap_q': 0.75,
        'vix_q': 0.50,
        'realized_vol_q': 0.70,
        'drawdown_q': 0.45,
        'max_equity_drawdown': 0.10,
        'note': 'Conservative high-conviction trend state.',
    },
    {
        'candidate': 'trend_q70_floor45_noeqdd',
        'enabled': True,
        'min_target': 0.45,
        'max_target': 0.55,
        'max_base_position': 0.35,
        'anchor_q': 0.70,
        'mom_q': 0.70,
        'trend_slope_q': 0.65,
        'trend_gap_q': 0.65,
        'vix_q': 0.50,
        'realized_vol_q': 0.75,
        'drawdown_q': 0.40,
        'max_equity_drawdown': None,
        'note': 'Diagnostic lift without path-equity drawdown guard.',
    },
]

EXTERNAL_SOURCES = [
    'https://github.com/microsoft/qlib',
    'https://arxiv.org/abs/2009.11189',
    'https://doi.org/10.21314/JCF.2016.322',
    'https://doi.org/10.1016/j.jfineco.2011.11.003',
    'https://doi.org/10.1016/j.jfineco.2016.05.002',
    'https://doi.org/10.1111/jofi.12513',
    'https://github.com/AI4Finance-Foundation/FinRL',
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return str(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + chr(10), encoding='utf-8')

print('root', ROOT)
print('card', CARD)


root /Users/rziiiii/.openclaw/workspace
card /Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/trend_participation_guard_screen_v1


In [2]:
profile = load_profile(PROFILE_NAME)
init_qlib(QLIB_DATA_PATH)

fold_features: dict[str, pd.DataFrame] = {}
fold_train_features: dict[str, pd.DataFrame] = {}
base_daily_paths: dict[str, pd.DataFrame] = {}

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    full = pd.concat([
        segment_frame(dataset, 'train'),
        segment_frame(dataset, 'valid'),
        segment_frame(dataset, 'test'),
    ]).sort_index()
    full = full.reset_index().rename(columns={'index': 'datetime'})
    full['datetime'] = pd.to_datetime(full['datetime'])
    price = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index().reset_index()
    price['datetime'] = pd.to_datetime(price['datetime'])
    full = full.merge(price, on='datetime', how='left')
    fold_features[fold_name] = full
    train_start, train_end = map(pd.Timestamp, spec['train'])
    fold_train_features[fold_name] = full[full['datetime'].between(train_start, train_end)].copy()

    daily = pd.read_csv(spec['daily'])
    daily['datetime'] = pd.to_datetime(daily['datetime'])
    if 'strategy_nav' not in daily.columns:
        if 'account' in daily.columns:
            daily['strategy_nav'] = pd.to_numeric(daily['account'], errors='coerce')
        elif 'nav' in daily.columns:
            daily['strategy_nav'] = pd.to_numeric(daily['nav'], errors='coerce')
    daily = daily.merge(price, on='datetime', how='left')
    if 'buyhold_nav' not in daily.columns:
        first_nav = float(pd.to_numeric(daily['strategy_nav'], errors='coerce').dropna().iloc[0])
        first_price = float(pd.to_numeric(daily['price'], errors='coerce').dropna().iloc[0])
        daily['buyhold_nav'] = first_nav * pd.to_numeric(daily['price'], errors='coerce') / first_price
    base_daily_paths[fold_name] = daily

pd.DataFrame([
    {
        'fold': fold,
        'feature_rows': len(fold_features[fold]),
        'train_rows': len(fold_train_features[fold]),
        'daily_rows': len(base_daily_paths[fold]),
        'daily_start': str(base_daily_paths[fold]['datetime'].min().date()),
        'daily_end': str(base_daily_paths[fold]['datetime'].max().date()),
    }
    for fold in WALK_FORWARD_WINDOWS
])


[43203:MainThread](2026-05-26 12:58:46,249) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[43203:MainThread](2026-05-26 12:58:47,039) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[43203:MainThread](2026-05-26 12:58:47,040) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[43203:MainThread](2026-05-26 12:58:49,151) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[43203:MainThread](2026-05-26 12:58:49,153) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[43203:MainThread](2026-05-26 12:58:49,157) INFO - qlib.timer - [log.py:127] - Time cost: 0.007s | Init data Done


[43203:MainThread](2026-05-26 12:58:51,507) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[43203:MainThread](2026-05-26 12:58:51,510) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[43203:MainThread](2026-05-26 12:58:51,512) INFO - qlib.timer - [log.py:127] - Time cost: 0.006s | Init data Done


,fold,feature_rows,train_rows,daily_rows,daily_start,daily_end
0,wf_2022,2973,2470,250,2022-01-03,2022-12-29
1,wf_2023_2024,3475,2973,251,2024-01-02,2024-12-30


Candidate thresholds are derived only from each fold's train segment. The 2024 diagnostic features define the feature family, but not the concrete thresholds.

In [3]:
def qvalue(train: pd.DataFrame, column: str, q: float) -> float:
    return float(pd.to_numeric(train[column], errors='coerce').quantile(float(q)))


def config_from_recipe(recipe: dict[str, Any], train: pd.DataFrame) -> TrendParticipationGuardConfig:
    if not recipe.get('enabled', False):
        return TrendParticipationGuardConfig(enabled=False, max_equity_drawdown=None)
    return TrendParticipationGuardConfig(
        enabled=True,
        min_target=float(recipe['min_target']),
        max_target=float(recipe['max_target']),
        max_base_position=float(recipe['max_base_position']),
        min_anchor_gap_180=qvalue(train, 'anchor_gap_180', recipe['anchor_q']),
        min_mom_63=qvalue(train, 'mom_63', recipe['mom_q']),
        min_trend_slope_63=qvalue(train, 'trend_slope_63', recipe['trend_slope_q']),
        min_trend_gap_63=qvalue(train, 'trend_gap_63', recipe['trend_gap_q']),
        max_vix_percentile_252=qvalue(train, 'vix_percentile_252', recipe['vix_q']),
        max_realized_vol_63=qvalue(train, 'realized_vol_63', recipe['realized_vol_q']),
        min_drawdown_63=qvalue(train, 'drawdown_63', recipe['drawdown_q']),
        max_equity_drawdown=recipe['max_equity_drawdown'],
    )

config_rows = []
for fold_name, train in fold_train_features.items():
    for recipe in CANDIDATE_RECIPES:
        cfg = config_from_recipe(recipe, train)
        row = {
            'fold': fold_name,
            'candidate': recipe['candidate'],
            'enabled': bool(recipe.get('enabled', False)),
            'note': recipe.get('note', ''),
            **cfg.__dict__,
        }
        config_rows.append(row)

candidate_configs = pd.DataFrame(config_rows)
candidate_configs.to_csv(CARD / 'candidate_configs.csv', index=False)
candidate_configs


,fold,candidate,enabled,note,min_target,max_target,max_base_position,min_anchor_gap_180,min_mom_63,min_trend_slope_63,min_trend_gap_63,max_vix_percentile_252,max_realized_vol_63,min_drawdown_63,max_equity_drawdown
0,wf_2022,base_seq_cap10_two_of_three,False,Unchanged selected warm-start action-cap path.,0.45,0.65,0.35,0.500000,0.400000,0.100000,0.100000,0.450000,1.000000,-0.250000,NaN
1,wf_2022,trend_q60_floor45_guard15,True,Broad train-quantile trend state with equity g...,0.45,0.55,0.35,0.245745,0.257826,0.069499,0.076901,0.523810,0.854766,-0.172098,0.15
2,wf_2022,trend_q70_floor45_guard10,True,Stricter low-vol trend state with tighter equi...,0.45,0.55,0.35,0.303922,0.309961,0.077194,0.088486,0.468254,0.831833,-0.149834,0.10
3,wf_2022,trend_q70_floor55_guard15,True,Higher participation floor with moderate equit...,0.55,0.65,0.35,0.303922,0.309961,0.077194,0.088486,0.523810,0.854766,-0.172098,0.15
4,wf_2022,trend_q80_floor55_guard10,True,Conservative high-conviction trend state.,0.55,0.65,0.30,0.391388,0.381433,0.094161,0.113315,0.468254,0.805250,-0.124128,0.10
5,wf_2022,trend_q70_floor45_noeqdd,True,Diagnostic lift without path-equity drawdown g...,0.45,0.55,0.35,0.303922,0.309961,0.077194,0.088486,0.468254,0.831833,-0.149834,NaN
6,wf_2023_2024,base_seq_cap10_two_of_three,False,Unchanged selected warm-start action-cap path.,0.45,0.65,0.35,0.500000,0.400000,0.100000,0.100000,0.450000,1.000000,-0.250000,NaN
7,wf_2023_2024,trend_q60_floor45_guard15,True,Broad train-quantile trend state with equity g...,0.45,0.55,0.35,0.236524,0.247982,0.064541,0.071094,0.521825,0.965539,-0.194419,0.15
8,wf_2023_2024,trend_q70_floor45_guard10,True,Stricter low-vol trend state with tighter equi...,0.45,0.55,0.35,0.297046,0.304011,0.073642,0.082859,0.464286,0.884659,-0.168598,0.10
9,wf_2023_2024,trend_q70_floor55_guard15,True,Higher participation floor with moderate equit...,0.55,0.65,0.35,0.297046,0.304011,0.073642,0.082859,0.521825,0.965539,-0.194419,0.15


In [4]:
def scored_path_for_candidate(fold_name: str, recipe: dict[str, Any]) -> pd.DataFrame:
    daily = base_daily_paths[fold_name].copy()
    daily['datetime'] = pd.to_datetime(daily['datetime'])
    if not recipe.get('enabled', False):
        out = daily.copy()
        out['base_position'] = out['position']
        out['base_strategy_nav'] = out['strategy_nav']
        out['trend_participation_signal_ready'] = False
        out['trend_participation_guard_ready'] = False
        out['trend_participation_equity_allowed'] = True
        out['trend_participation_ready'] = False
        out['trend_participation_lifted'] = False
        out['equity_nav'] = out['strategy_nav']
        out['equity_drawdown'] = 0.0
        return out
    cfg = config_from_recipe(recipe, fold_train_features[fold_name])
    return apply_trend_participation_guard_overlay(
        daily,
        fold_features[fold_name],
        cfg,
        position_col='position',
        price_col='price',
    )

path_manifest_rows = []
window_rows = []
cluster_rows = []
daily_paths: dict[tuple[str, str], pd.DataFrame] = {}
metrics_by_candidate: dict[str, dict[str, dict[str, Any]]] = {}
cluster_metrics_by_candidate: dict[str, dict[str, dict[str, Any]]] = {}
poor63 = pd.read_csv(SOURCE_STATE_DIAG / 'poor_windows_63.csv').head(3)

for recipe in CANDIDATE_RECIPES:
    candidate = recipe['candidate']
    metrics_by_candidate[candidate] = {}
    cluster_metrics_by_candidate[candidate] = {}
    for fold_name in WALK_FORWARD_WINDOWS:
        path = scored_path_for_candidate(fold_name, recipe)
        path.insert(0, 'fold', fold_name)
        path.insert(1, 'candidate', candidate)
        daily_paths[(candidate, fold_name)] = path
        metrics = daily_path_metrics(path)
        metrics_by_candidate[candidate][fold_name] = metrics
        window_rows.append({'candidate': candidate, 'fold': fold_name, **metrics})
        path_manifest_rows.append({
            'candidate': candidate,
            'fold': fold_name,
            'rows': len(path),
            'ready_days': int(path['trend_participation_ready'].sum()),
            'lifted_days': int(path['trend_participation_lifted'].sum()),
            'avg_position_lift': float((path['position'] - path['base_position']).mean()),
            'max_position_lift': float((path['position'] - path['base_position']).max()),
            'equity_blocked_days': int((~path['trend_participation_equity_allowed']).sum()),
        })
        if fold_name == 'wf_2023_2024':
            for rank, row in poor63.reset_index(drop=True).iterrows():
                cluster_name = f'wf2024_poor63_rank{rank + 1}'
                c_metrics = slice_daily_path_metrics(path, start=str(row['start']), end=str(row['end']))
                cluster_metrics_by_candidate[candidate][cluster_name] = c_metrics
                cluster_rows.append({'candidate': candidate, 'cluster': cluster_name, **c_metrics})

window_metrics = pd.DataFrame(window_rows)
cluster_metrics = pd.DataFrame(cluster_rows)
path_manifest = pd.DataFrame(path_manifest_rows)
window_metrics.to_csv(CARD / 'window_metrics.csv', index=False)
cluster_metrics.to_csv(CARD / 'cluster_metrics.csv', index=False)
path_manifest.to_csv(CARD / 'candidate_path_manifest.csv', index=False)

window_metrics.sort_values(['fold', 'relative_return', 'max_dd'], ascending=[True, False, True])


,candidate,fold,rows,start,end,return,buy_and_hold_return,relative_return,max_dd,avg_pos
0,base_seq_cap10_two_of_three,wf_2022,250,2022-01-03,2022-12-29,-9.287319,-86.588072,77.300753,13.280734,0.076032
8,trend_q80_floor55_guard10,wf_2022,250,2022-01-03,2022-12-29,-10.835876,-86.588072,75.752195,14.452165,0.076032
2,trend_q60_floor45_guard15,wf_2022,250,2022-01-03,2022-12-29,-15.369300,-86.588072,71.218772,18.171048,0.079632
4,trend_q70_floor45_guard10,wf_2022,250,2022-01-03,2022-12-29,-15.369300,-86.588072,71.218772,18.171048,0.079632
10,trend_q70_floor45_noeqdd,wf_2022,250,2022-01-03,2022-12-29,-15.369300,-86.588072,71.218772,18.171048,0.079632
6,trend_q70_floor55_guard15,wf_2022,250,2022-01-03,2022-12-29,-16.369768,-86.588072,70.218304,19.138395,0.080432
1,base_seq_cap10_two_of_three,wf_2023_2024,251,2024-01-02,2024-12-30,2.372261,-0.356635,2.728896,20.232446,0.160573
9,trend_q80_floor55_guard10,wf_2023_2024,251,2024-01-02,2024-12-30,-4.462797,-0.356635,-4.106162,23.713655,0.197654
3,trend_q60_floor45_guard15,wf_2023_2024,251,2024-01-02,2024-12-30,-5.259420,-0.356635,-4.902785,29.143110,0.216943
5,trend_q70_floor45_guard10,wf_2023_2024,251,2024-01-02,2024-12-30,-10.469217,-0.356635,-10.112582,24.243495,0.200290


In [5]:
path_manifest.sort_values(['fold', 'lifted_days', 'avg_position_lift'], ascending=[True, False, False])


,candidate,fold,rows,ready_days,lifted_days,avg_position_lift,max_position_lift,equity_blocked_days
6,trend_q70_floor55_guard15,wf_2022,250,2,2,0.004400,0.550000,61
2,trend_q60_floor45_guard15,wf_2022,250,2,2,0.003600,0.450000,38
4,trend_q70_floor45_guard10,wf_2022,250,2,2,0.003600,0.450000,170
10,trend_q70_floor45_noeqdd,wf_2022,250,2,2,0.003600,0.450000,0
0,base_seq_cap10_two_of_three,wf_2022,250,0,0,0.000000,0.000000,0
8,trend_q80_floor55_guard10,wf_2022,250,0,0,0.000000,0.000000,69
3,trend_q60_floor45_guard15,wf_2023_2024,251,49,49,0.056371,0.387769,144
7,trend_q70_floor55_guard15,wf_2023_2024,251,46,46,0.071547,0.487769,180
5,trend_q70_floor45_guard10,wf_2023_2024,251,34,34,0.039717,0.387769,141
11,trend_q70_floor45_noeqdd,wf_2023_2024,251,34,34,0.039717,0.387769,0


In [6]:
cluster_metrics.sort_values(['cluster', 'relative_return', 'max_dd'], ascending=[True, False, True])


,candidate,cluster,rows,start,end,return,buy_and_hold_return,relative_return,max_dd,avg_pos
0,base_seq_cap10_two_of_three,wf2024_poor63_rank1,63,2024-04-22,2024-07-22,0.999167,65.045216,-64.046050,12.867847,0.218922
3,trend_q60_floor45_guard15,wf2024_poor63_rank1,63,2024-04-22,2024-07-22,-1.704174,65.045216,-66.749390,12.772524,0.223089
6,trend_q70_floor45_guard10,wf2024_poor63_rank1,63,2024-04-22,2024-07-22,-1.704174,65.045216,-66.749390,12.772524,0.223089
15,trend_q70_floor45_noeqdd,wf2024_poor63_rank1,63,2024-04-22,2024-07-22,-1.704174,65.045216,-66.749390,12.772524,0.223089
9,trend_q70_floor55_guard15,wf2024_poor63_rank1,63,2024-04-22,2024-07-22,-3.990607,65.045216,-69.035823,14.801499,0.224676
12,trend_q80_floor55_guard10,wf2024_poor63_rank1,63,2024-04-22,2024-07-22,-3.990607,65.045216,-69.035823,14.801499,0.224676
4,trend_q60_floor45_guard15,wf2024_poor63_rank2,63,2024-04-16,2024-07-16,4.080754,61.075485,-56.994731,8.125503,0.221243
10,trend_q70_floor55_guard15,wf2024_poor63_rank2,63,2024-04-16,2024-07-16,4.080754,61.075485,-56.994731,8.125503,0.222831
7,trend_q70_floor45_guard10,wf2024_poor63_rank2,63,2024-04-16,2024-07-16,4.080754,61.075485,-56.994731,8.125503,0.221243
16,trend_q70_floor45_noeqdd,wf2024_poor63_rank2,63,2024-04-16,2024-07-16,4.080754,61.075485,-56.994731,8.125503,0.221243


In [7]:
objective_config = StateConditionedObjectiveConfig(
    target_relative_return=TARGET_RELATIVE_RETURN,
    target_max_dd=TARGET_MAX_DD,
    cluster_relative_floor=-30.0,
    cluster_participation_floor=0.35,
    cluster_max_dd_gap_weight=5.0,
    cluster_participation_gap_weight=100.0,
)
scores = score_state_conditioned_objective(
    metrics_by_candidate,
    cluster_metrics_by_candidate=cluster_metrics_by_candidate,
    config=objective_config,
)
summary_base = summarize_state_conditioned_objective(scores)
review_md = render_state_conditioned_objective_review(scores, summary_base)
(CARD / 'objective_review.md').write_text(review_md, encoding='utf-8')
scores.to_csv(CARD / 'objective_scores.csv', index=False)
display(Markdown(review_md))
scores


# State-Conditioned Objective Screen

## Decision

- decision: `no_objective_pass_candidate`
- selected_candidate: `base_seq_cap10_two_of_three`
- next_mechanism: `train_state_conditioned_multi_objective`

## Ranking

| candidate | pass | loss | wf pass | cluster pass | rel loss | dd loss | part loss | worst wf | worst cluster |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- | --- |
| base_seq_cap10_two_of_three | False | 216.92 | 1/2 | False | 120.49 | 52.32 | 44.10 | wf_2023_2024 | wf2024_poor63_rank1 |
| trend_q80_floor55_guard10 | False | 245.60 | 1/2 | False | 129.71 | 87.14 | 28.75 | wf_2023_2024 | wf2024_poor63_rank1 |
| trend_q70_floor45_guard10 | False | 301.12 | 0/2 | False | 140.58 | 132.07 | 28.48 | wf_2023_2024 | wf2024_poor63_rank1 |
| trend_q70_floor45_noeqdd | False | 301.12 | 0/2 | False | 140.58 | 132.07 | 28.48 | wf_2023_2024 | wf2024_poor63_rank1 |
| trend_q60_floor45_guard15 | False | 332.16 | 0/2 | False | 125.52 | 181.07 | 25.57 | wf_2023_2024 | wf2024_poor63_rank1 |
| trend_q70_floor55_guard15 | False | 400.33 | 0/2 | False | 138.73 | 236.34 | 25.25 | wf_2023_2024 | wf2024_poor63_rank1 |

## Readout

- A passing row is a promotion-review input, not an automatic runtime bind.
- If no row passes, treat the selected row as a diagnostic target and keep no runtime promotion.
- Required terms should be implemented in training-time reward/checkpoint selection before another path-level threshold search.


,candidate,window_count,window_pass_count,window_gate,cluster_count,cluster_gate,objective_pass,objective_loss,relative_loss,drawdown_loss,participation_loss,min_relative_return,max_max_dd,mean_avg_pos,worst_window,worst_cluster
0,base_seq_cap10_two_of_three,2,1,False,3,False,False,216.918189,120.492417,52.324464,44.101308,2.728896,20.232446,0.118302,wf_2023_2024,wf2024_poor63_rank1
1,trend_q80_floor55_guard10,2,1,False,3,False,False,245.603150,129.714559,87.136551,28.752040,-4.106162,23.713655,0.136843,wf_2023_2024,wf2024_poor63_rank1
2,trend_q70_floor45_guard10,2,0,False,3,False,False,301.124354,140.575412,132.073045,28.475896,-10.112582,24.243495,0.139961,wf_2023_2024,wf2024_poor63_rank1
3,trend_q70_floor45_noeqdd,2,0,False,3,False,False,301.124354,140.575412,132.073045,28.475896,-10.112582,24.243495,0.139961,wf_2023_2024,wf2024_poor63_rank1
4,trend_q60_floor45_guard15,2,0,False,3,False,False,332.159314,125.523351,181.069192,25.566771,-4.902785,29.143110,0.148287,wf_2023_2024,wf2024_poor63_rank1
5,trend_q70_floor55_guard15,2,0,False,3,False,False,400.326058,138.734515,236.342232,25.249311,-11.679192,33.461230,0.156276,wf_2023_2024,wf2024_poor63_rank1


In [8]:
top = scores.iloc[0].to_dict() if not scores.empty else {}
selected_candidate = str(top.get('candidate', ''))
for fold_name in WALK_FORWARD_WINDOWS:
    if (selected_candidate, fold_name) in daily_paths:
        daily_paths[(selected_candidate, fold_name)].to_csv(CARD / f'{selected_candidate}_{fold_name}_daily_path.csv', index=False)

objective_pass = bool(top.get('objective_pass', False)) if top else False
if objective_pass:
    decision = 'trend_participation_guard_two_fold_objective_pass_no_promotion_pending_full_wf'
    next_mechanism = 'validate_fixed_split_wf2020_and_checkpoint_target_scope_before_training'
    blocker = 'Only wf_2022/wf_2023_2024 and prior poor-window clusters were screened; fixed split, wf_2020, and leakage guard remain unproven.'
else:
    decision = 'trend_participation_guard_no_objective_pass_no_promotion'
    next_mechanism = 'encode_fold_local_trend_participation_with_drawdown_guard_as_training_or_checkpoint_objective'
    blocker = 'No candidate jointly cleared +30pp relative return, <=15% MaxDD, and poor-window cluster participation gates.'

selected_window = window_metrics[window_metrics['candidate'].eq(selected_candidate)].to_dict(orient='records') if selected_candidate else []
selected_clusters = cluster_metrics[cluster_metrics['candidate'].eq(selected_candidate)].to_dict(orient='records') if selected_candidate else []
selected_manifest = path_manifest[path_manifest['candidate'].eq(selected_candidate)].to_dict(orient='records') if selected_candidate else []

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'role': 'diagnose',
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'warm_start_action_cap_validation_v1:seq_cap10_two_of_three',
    'profile': PROFILE_NAME,
    'changed_layer': ['model_selection', 'counterfactual_overlay'],
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'selected_candidate': selected_candidate,
    'selected_objective_loss': float(top.get('objective_loss', 0.0)) if top else None,
    'selected_window_gate': bool(top.get('window_gate', False)) if top else False,
    'selected_cluster_gate': bool(top.get('cluster_gate', False)) if top else False,
    'selected_worst_window': str(top.get('worst_window', '')) if top else '',
    'selected_worst_cluster': str(top.get('worst_cluster', '')) if top else '',
    'promotion_allowed': False,
    'training_allowed_from_this_card': False,
    'promotion_blocker': blocker,
    'next_mechanism': next_mechanism,
    'external_sources': EXTERNAL_SOURCES,
    'selected_window_metrics': selected_window,
    'selected_cluster_metrics': selected_clusters,
    'selected_path_manifest': selected_manifest,
    'sidecars': [
        'candidate_configs.csv',
        'window_metrics.csv',
        'cluster_metrics.csv',
        'candidate_path_manifest.csv',
        'objective_scores.csv',
        'objective_review.md',
        'selection.json',
        'summary.json',
    ] + ([f'{selected_candidate}_{fold}_daily_path.csv' for fold in WALK_FORWARD_WINDOWS] if selected_candidate else []),
}

summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'selected_candidate': selected_candidate,
    'selected_objective_loss': selection['selected_objective_loss'],
    'selected_window_gate': selection['selected_window_gate'],
    'selected_cluster_gate': selection['selected_cluster_gate'],
    'selected_worst_window': selection['selected_worst_window'],
    'selected_worst_cluster': selection['selected_worst_cluster'],
    'next_mechanism': next_mechanism,
    'promotion_allowed': False,
    'training_allowed_from_this_card': False,
    'conclusion': [
        'Trend-participation guard candidates were derived from fold-local train quantiles; 2024 labels were used only for scoring diagnostic paths.',
        'This card is a counterfactual screen, not a runtime profile promotion and not an actor-retention artifact.',
        'If no objective-pass candidate appears, the best row should be converted into a fold-local training/checkpoint objective rather than another runtime threshold overlay.',
    ],
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
print(json.dumps(selection, indent=2))


{
  "experiment": "trend_participation_guard_screen_v1",
  "created_at": "2026-05-26T04:58:52.075773+00:00",
  "decision": "trend_participation_guard_no_objective_pass_no_promotion",
  "role": "diagnose",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "warm_start_action_cap_validation_v1:seq_cap10_two_of_three",
  "profile": "sparse_release_bc_ppo_screen_v1",
  "changed_layer": [
    "model_selection",
    "counterfactual_overlay"
  ],
  "target_relative_return_pp": 30.0,
  "target_max_dd_pct": 15.0,
  "selected_candidate": "base_seq_cap10_two_of_three",
  "selected_objective_loss": 216.91818920030946,
  "selected_window_gate": false,
  "selected_cluster_gate": false,
  "selected_worst_window": "wf_2023_2024",
  "selected_worst_cluster": "wf2024_poor63_rank1",
  "promotion_allowed": false,
  "training_allowed_from_this_card": false,
  "promotion_blocker": "No candidate jointly cleared +30pp relative return, <=15% MaxDD, and poor-window 

In [9]:
selected = json.loads((CARD / 'selection.json').read_text(encoding='utf-8'))
scores = pd.read_csv(CARD / 'objective_scores.csv')
windows = pd.read_csv(CARD / 'window_metrics.csv')
manifest = pd.read_csv(CARD / 'candidate_path_manifest.csv')
base_rows = windows[windows['candidate'].eq('base_seq_cap10_two_of_three')].set_index('fold')
non_base = scores[~scores['candidate'].eq('base_seq_cap10_two_of_three')]
best_overlay = non_base.iloc[0].to_dict() if not non_base.empty else {}
base_2024 = base_rows.loc['wf_2023_2024'].to_dict()
best_overlay_2024 = windows[
    windows['candidate'].eq(best_overlay.get('candidate', ''))
    & windows['fold'].eq('wf_2023_2024')
]
best_overlay_2024 = best_overlay_2024.iloc[0].to_dict() if len(best_overlay_2024) else {}
ready_2024 = manifest[
    manifest['candidate'].eq(best_overlay.get('candidate', ''))
    & manifest['fold'].eq('wf_2023_2024')
]
ready_2024 = ready_2024.iloc[0].to_dict() if len(ready_2024) else {}
conclusion_md = f"""
## Runtime Conclusion

Decision: `{selected['decision']}`. Promotion remains blocked and no actor is retained.

Selected row by objective loss: `{selected['selected_candidate']}` with objective loss `{selected['selected_objective_loss']:.2f}`. It is the unchanged warm-start `seq_cap10_two_of_three` path, not a trend-floor overlay.

Base fold metrics: `wf_2022` relative `{base_rows.loc['wf_2022', 'relative_return']:.2f}pp`, MaxDD `{base_rows.loc['wf_2022', 'max_dd']:.2f}%`; `wf_2023_2024` relative `{base_2024['relative_return']:.2f}pp`, MaxDD `{base_2024['max_dd']:.2f}%`.

Best non-base overlay: `{best_overlay.get('candidate', '')}`. In `wf_2023_2024` it produced relative `{best_overlay_2024.get('relative_return', float('nan')):.2f}pp`, MaxDD `{best_overlay_2024.get('max_dd', float('nan')):.2f}%`, ready days `{int(ready_2024.get('ready_days', 0))}`, and lifted days `{int(ready_2024.get('lifted_days', 0))}`. That is worse than the base path despite adding trend participation.

Interpretation: the 2024 underparticipation window is not fixed by a post-hoc runtime trend floor. The lift arrives in the wrong path context and increases drawdown / negative excess. The next valid step is to encode the fold-local trend-participation state as a training or checkpoint-selection objective with an explicit drawdown/volatility term, not to continue a runtime threshold grid.

Next action: `{selected['next_mechanism']}`.
""".strip()
display(Markdown(conclusion_md))


## Runtime Conclusion

Decision: `trend_participation_guard_no_objective_pass_no_promotion`. Promotion remains blocked and no actor is retained.

Selected row by objective loss: `base_seq_cap10_two_of_three` with objective loss `216.92`. It is the unchanged warm-start `seq_cap10_two_of_three` path, not a trend-floor overlay.

Base fold metrics: `wf_2022` relative `77.30pp`, MaxDD `13.28%`; `wf_2023_2024` relative `2.73pp`, MaxDD `20.23%`.

Best non-base overlay: `trend_q80_floor55_guard10`. In `wf_2023_2024` it produced relative `-4.11pp`, MaxDD `23.71%`, ready days `24`, and lifted days `24`. That is worse than the base path despite adding trend participation.

Interpretation: the 2024 underparticipation window is not fixed by a post-hoc runtime trend floor. The lift arrives in the wrong path context and increases drawdown / negative excess. The next valid step is to encode the fold-local trend-participation state as a training or checkpoint-selection objective with an explicit drawdown/volatility term, not to continue a runtime threshold grid.

Next action: `encode_fold_local_trend_participation_with_drawdown_guard_as_training_or_checkpoint_objective`.